In [27]:
import pandas as pd
import numpy as np
from gensim.models import Word2Vec
from matplotlib import pyplot as plt
from scipy.cluster.hierarchy import dendrogram
from sklearn.datasets import load_iris
from sklearn.cluster import AgglomerativeClustering
from gensim.test.utils import common_texts
import warnings
warnings.filterwarnings('ignore')

In [2]:
data = pd.read_feather('../Datasets/dataset.ftr')

In [3]:
df = data

In [4]:
data

,twit_id,date,time,twit,refined_twit,corrected_twit,okt
0,1553510754581487616,2022-07-30,22:40:01,백신 4차 노바백스로 맞음맞으면서 부작용 사례 물어보니아직 접수 된 건이 하나도 없...,백신 4차 노바백스로 맞음맞으면서 부작용 사례 물어보니아직 접수 된 건이 하나도 없...,백신 4차 노바백스로 맞으면 맞으면서 부작용 사례 물어보니 아직 접수된 건이 하나도...,"[부작용, 사례, 접수, 부위, 통증, 확실하다, 부작용]"
1,1553293555854061571,2022-07-30,08:16:57,노바백스 후기화이자 보다 안 아픔 근데 팔 아파서 잘 못 움직임 근데 이것도 케바케...,노바백스 후기화이자 보다 안 아픔 근데 팔 아파서 잘 못 움직임 근데 이것도 케바케...,노바백스 후기 화이자보다안 아픔 근데 팔 아파서 잘 못 움직임 근데 이것도 때에 따...,"[아픔, 아프다, 움직임, 할머니, 엄마, 아프다]"
2,1552540061635293184,2022-07-28,06:22:50,"1,2차 화이자3차 노바백스생명의 위협을 느껴서 이제서야 3차 맞음근데 15분 있다...",12차 화이자3차 노바백스생명의 위협을 느껴서 이제서야 3차 맞음근데 15분 있다가...,12차 화이자 3차 노바백스 생명의 위협을 느껴서 인제야 3차 맞음 근데 15분 있...,"[생명, 위협, 인제, 가라, 스스로]"
3,1551897969049235461,2022-07-26,11:51:23,백신 3차접종 완료자만 입국후 격리면제 가능인정백신 2차 접종까지 지정 백신: 화이...,백신 3차접종 완료자만 입국후 격리면제 가능인정백신 2차 접종까지 지정 백신 화이자...,백신 3차 접종 완료자만 입국 후 격리 면제 가능 인정 백신 2차 접종까지 지정 백...,"[완료, 자만, 입국, 격리, 면제, 가능, 인정, 지정, 지정, 일본, 확진, 완..."
4,1551731672789315586,2022-07-26,00:50:35,@ahomoo1 1차 2차 바이러스벡터 백신3차 mRNA 백신.4차는 유전자 재조합...,1차 2차 바이러스벡터 백신3차 mrna 백신.4차는 유전자 재조합 백신인 노바백스...,1차 2차 바이러스벡터 백신 3차 mRNA 백신. 4차는 유전자 재조합 백신인 노바...,"[바이러스, 벡터, 유전자, 조합]"
...,...,...,...,...,...,...,...
118771,1356238114646151168,2021-02-01,13:48:58,"화이자-바이오엔테크, 코로나 백신 생산 확대로 EU에 7,500만회분 추가 공급: ...",화이자바이오엔테크 코로나백신 생산 확대로 eu에 7500만회분 추가 공급 당초 계회...,"화이자 바이오엔테크 코로나백신 생산 확대로 EU에 7,500만 회분 추가 공급 당초...","[바이오엔테크, 생산, 확대, 추가, 공급, 당초, 계획, 규모]"
118772,1356195135411773440,2021-02-01,10:58:11,"화이자 백신접종과정 넘모 비효율적..해동해서 해동한 백신에 식염수 섞고, 식염수 섞...",화이자접종과정 너무모 비효율적..해동해서 해동한 백신에 식염수 섞고 식염수 섞은 백...,화이자 접종 과정 너무 모 비효율적…. 해동해서 해동한 백신에 식염수 섞고 식염수 ...,"[과정, 효율, 해동, 해동, 식염수, 식염수, 등분, 해동, 호들갑, 단위, 지고]"
118773,1356171631387762690,2021-02-01,09:24:48,난 화이자 백신을 맞을 생각이 1나노드램도 없어.내가 이해하는 게 맞는지는 잘 모르...,난 화이자을 맞을 생각이 1나노드램도 없어.내가 이해하는 게 맞는지는 잘 모르겠다만...,난 화이자를 맞을 생각이 1나노그램도 없어. 내가 이해하는 게 맞는지는 잘 모르겠지...,"[나노, 이해, 유전자, 줄기, 항체, 정보, 만화, 저항, 만만, 음식, 유전자,..."
118774,1356169358980993026,2021-02-01,09:15:46,@KigOF_TK 여러분들 화장실 알아요 나도 알아요 근데 중요한거 fucki 있어...,여러분들 화장실 알아요 나도 알아요 근데 중요한거 fucki 있어요 잘들어요 화장을...,여러분들 화장실 알아요. 나도 알아요. 근데 중요한 거 fuck 있어요. 잘 들어요...,"[화장실, 중요하다, 화장, 얼굴, 미안하다, 화장, 고치, 바이러스]"


In [36]:
train_data =[]
for row in df.okt:
    train_data.append(row.tolist())

In [37]:
model = Word2Vec(sentences=train_data ,min_count=0)

In [38]:
model.save('../models/word2vec.model') 